In [34]:
from collections import namedtuple
import math
from collections import defaultdict
import random
import random

In [35]:
class Game:
    """A game is similar to a problem, but it has a terminal test instead of 
    a goal test, and a utility for each terminal state. To create a game, 
    subclass this class and implement `actions`, `result`, `is_terminal`, 
    and `utility`. You will also need to set the .initial attribute to the 
    initial state; this can be done in the constructor."""

    def actions(self, state):
        """Return a collection of the allowable moves from this state."""
        raise NotImplementedError

    def result(self, state, move):
        """Return the state that results from making a move from a state."""
        raise NotImplementedError

    def is_terminal(self, state):
        """Return True if this is a final state for the game."""
        return not self.actions(state)
    
    def utility(self, state, player):
        """Return the value of this final state to player."""
        raise NotImplementedError

    def terminal_test(self, state):
        return self.is_terminal(state)

    def to_move(self,state):
        return state.to_move

In [36]:
class Board(defaultdict):
    """A board has the player to move, a cached utility value, 
    and a dict of {(x, y): player} entries, where player is 'X' or 'O'."""
    empty = '.'
    off = '#'
    
    def __init__(self, width=8, height=8, to_move=None, **kwds):
        self.__dict__.update(width=width, height=height, to_move=to_move, **kwds)
        
    def new(self, changes: dict, **kwds) -> 'Board':
        "Given a dict of {(x, y): contents} changes, return a new Board with the changes."
        board = Board(width=self.width, height=self.height, **kwds)
        board.update(self)
        board.update(changes)
        return board

    def __missing__(self, loc):
        x, y = loc
        if 0 <= x < self.width and 0 <= y < self.height:
            return self.empty
        else:
            return self.off
            
    def __hash__(self): 
        return hash(tuple(sorted(self.items()))) + hash(self.to_move)
    
    def __repr__(self):
        def row(y): return ' '.join(self[x, y] for x in range(self.width))
        return '\n'.join(map(row, range(self.height))) +  '\n'

In [37]:

class TwoMoveTicTacToe(Game):  # Renamed from TicTacToe
    """Play TicTacToe on an `height` by `width` board, needing `k` in a row to win.
    'X' plays first against 'O'."""

    def __init__(self, height=3, width=3, k=3):
        self.k = k  # k in a row
        self.squares = {(x, y) for x in range(width) for y in range(height)}
        self.initial = Board(height=height, width=width, to_move='X', utility=0)

    def actions(self, board):
        """Legal moves are any square not yet taken."""
        return self.squares - set(board.keys())

    def result(self, board, square1, square2):
        """Place markers for the current player on two squares."""
        player = board.to_move
        board = board.new({square1: player, square2: player}, to_move=('O' if player == 'X' else 'X'))
        win1 = k_in_row(board, player, square1, self.k)
        win2 = k_in_row(board, player, square2, self.k)
        board.utility = (0 if not (win1 or win2) else +1 if player == 'X' else -1)
        return board

    def utility(self, board, player):
        """Return the value to player; 1 for win, -1 for loss, 0 otherwise."""
        return board.utility if player == 'X' else -board.utility

    def is_terminal(self, board):
        """A board is a terminal state if it is won or there are no empty squares."""
        return board.utility != 0 or len(self.squares) < 2

    def display(self, board): 
        print(board)

    def add_random_barrier(self, board):
        """Add a random barrier to the board."""
        player = board.to_move
        square = random.choice(list(self.actions(board))) if self.actions(board) else None
        if square is None:
            return board
        print('adding random barrier at', square)
        board = board.new({square: '#'}, to_move=board.to_move)
        win = k_in_row(board, player, square, self.k)
        board.utility = (0 if not win else +1 if player == 'X' else -1)
        return board


def k_in_row(board, player, square, k):
    """True if player has k pieces in a line through square."""
    if square is None:
        return False  # Return False if square is None
    def in_row(x, y, dx, dy): 
        return 0 if board[x, y] != player else 1 + in_row(x + dx, y + dy, dx, dy)
    return any(in_row(*square, dx, dy) + in_row(*square, -dx, -dy) - 1 >= k
               for (dx, dy) in ((0, 1), (1, 0), (1, 1), (1, -1)))

def __repr__(self):
    return self.__class__.__name__ + ' ' + str(dict(self)) 


In [38]:
def play_game(game, strategies: dict, verbose=False):
    """Play a turn-taking game where each player makes two moves per turn."""
    state = game.initial
    while not game.is_terminal(state):
        player = state.to_move
        try:
            move1, move2 = strategies[player](game, state)
            state = game.result(state, move1, move2)
            if verbose:
                print('Player', player, 'moves:', move1, 'and', move2)
                print(state)
            if move1 is None and move2 is None:
                raise ValueError("Invalid move: None")
        except ValueError as e:
           # print(f"Error: {e}")
            break
    return state

In [39]:
import time
random_player_time = 0 
def random_player(game, state):
    """Select two random moves for the current player."""
    global random_player_time
    start_time = time.time()
    move1, move2 = random_player_(game, state)
    elapsed_time = time.time() - start_time  # Calculate elapsed time
    random_player_time += elapsed_time
    return move1, move2

def random_player_(game, state):
    """Select two random moves for the current player."""
    legal_moves = list(game.actions(state))
    if len(legal_moves) < 2:
        raise ValueError("Not enough legal moves for two consecutive moves.")
    move1 = random.choice(legal_moves)
    legal_moves.remove(move1)  # Ensure the second move is different
    move2 = random.choice(legal_moves)
    return move1, move2

def player(search_algorithm):
    """A game player who uses the specified search algorithm"""
    return lambda game, state: search_algorithm(game, state)[1]

In [40]:
#add a global variable and capture the accumilated time minmax search took

minimax_time = 0
def minimax_search(game, state):
    """Search game tree to determine the best move; return (value, move) pair."""
    global minimax_time  # Use the global variable to track time
    start_time = time.time()  # Start the timer
    value, move = minimax_search_(game, state)
    elapsed_time = time.time() - start_time  # Calculate elapsed time
    minimax_time += elapsed_time  # Accumulate the time taken
    #print(f"Minimax search took {elapsed_time:.4f} seconds")
    return value, move

def minimax_search_(game, state):
    """Search game tree to determine the best two consecutive moves; return (value, (move1, move2)) pair."""
    start_time = time.time()  
    player = state.to_move

    def max_value(state):
        if game.is_terminal(state):
            return game.utility(state, player), (None, None)
        v, moves = -math.inf, (None, None)
        legal_moves = game.actions(state)
        for a1 in legal_moves:
            for a2 in legal_moves - {a1}:  # Ensure the second move is different
                v2, _ = min_value(game.result(state, a1, a2))
                if v2 > v:
                    v, moves = v2, (a1, a2)
        return v, moves

    def min_value(state):
        if game.is_terminal(state):
            return game.utility(state, player), (None, None)
        v, moves = +math.inf, (None, None)
        legal_moves = game.actions(state)
        for a1 in legal_moves:
            for a2 in legal_moves - {a1}:  # Ensure the second move is different
                v2, _ = max_value(game.result(state, a1, a2))
                if v2 < v:
                    v, moves = v2, (a1, a2)
        return v, moves
    
    return max_value(state)

In [41]:
minimax_time_alpha_beta = 0
def minimax_search_alpha_beta_prune(game, state):
    """Search game tree to determine the best two consecutive moves using alpha-beta pruning.
    Return (value, (move1, move2)) pair."""
    global minimax_time_alpha_beta  # Use the global variable to track time
    start_time = time.time()  # Start the timer
    value, move = minimax_search_alpha_beta_prune_(game, state)
    elapsed_time = time.time() - start_time  # Calculate elapsed time
    minimax_time_alpha_beta += elapsed_time  # Accumulate the time taken
    #print(f"Minimax search with alpha-beta pruning took {elapsed_time:.4f} seconds")
    return value, move

def minimax_search_alpha_beta_prune_(game, state):
    """Search game tree to determine the best two consecutive moves using alpha-beta pruning.
    Return (value, (move1, move2)) pair."""

    player = state.to_move

    def max_value(state, alpha, beta):
        if game.is_terminal(state):
            return game.utility(state, player), (None, None)
        v, moves = -math.inf, (None, None)
        legal_moves = list(game.actions(state))
        for a1 in legal_moves:
            for a2 in legal_moves:
                if a1 != a2:  # Ensure the two moves are distinct
                    v2, _ = min_value(game.result(state, a1, a2), alpha, beta)
                    if v2 > v:
                        v, moves = v2, (a1, a2)
                    if v >= beta:
                        return v, moves  # Beta cutoff
                    alpha = max(alpha, v)
        return v, moves

    def min_value(state, alpha, beta):
        if game.is_terminal(state):
            return game.utility(state, player), (None, None)
        v, moves = math.inf, (None, None)
        legal_moves = list(game.actions(state))
        for a1 in legal_moves:
            for a2 in legal_moves:
                if a1 != a2:  # Ensure the two moves are distinct
                    v2, _ = max_value(game.result(state, a1, a2), alpha, beta)
                    if v2 < v:
                        v, moves = v2, (a1, a2)
                    if v <= alpha:
                        return v, moves  # Alpha cutoff
                    beta = min(beta, v)
        return v, moves

    return max_value(state, -math.inf, math.inf)

In [42]:
minimax_time_alpha_beta_prune_hct = 0
def minimax_search_alpha_beta_prune_hct(game, state, depth_limit, heuristic_evaluation):
    """Search game tree to determine the best two consecutive moves using alpha-beta pruning with heuristic cutoff.
    Return (value, (move1, move2)) pair."""
    global minimax_time_alpha_beta_prune_hct  # Use the global variable to track time
    start_time = time.time()  # Start the timer
    value, move = minimax_search_alpha_beta_prune_hct_(game, state, depth_limit, heuristic_evaluation)
    elapsed_time = time.time() - start_time  # Calculate elapsed time
    minimax_time_alpha_beta_prune_hct += elapsed_time  # Accumulate the time taken
    #print(f"Minimax search with alpha-beta pruning and heuristic cutoff took {elapsed_time:.4f} seconds")
    return value, move

def minimax_search_alpha_beta_prune_hct_(game, state, depth_limit, heuristic_evaluation):
    """Search game tree to determine the best two consecutive moves using alpha-beta pruning with heuristic cutoff.
    Return (value, (move1, move2)) pair."""

    player = state.to_move

    def max_value(state, alpha, beta, depth):
        if game.is_terminal(state) or depth == 0:
            return heuristic_evaluation(game, state, player), (None, None)
        v, moves = -math.inf, (None, None)
        legal_moves = list(game.actions(state))
        for a1 in legal_moves:
            for a2 in legal_moves:
                if a1 != a2:  # Ensure the two moves are distinct
                    v2, _ = min_value(game.result(state, a1, a2), alpha, beta, depth - 1)
                    if v2 > v:
                        v, moves = v2, (a1, a2)
                    if v >= beta:
                        return v, moves  # Beta cutoff
                    alpha = max(alpha, v)
        return v, moves

    def min_value(state, alpha, beta, depth):
        if game.is_terminal(state) or depth == 0:
            return heuristic_evaluation(game, state, player), (None, None)
        v, moves = math.inf, (None, None)
        legal_moves = list(game.actions(state))
        for a1 in legal_moves:
            for a2 in legal_moves:
                if a1 != a2:  # Ensure the two moves are distinct
                    v2, _ = max_value(game.result(state, a1, a2), alpha, beta, depth - 1)
                    if v2 < v:
                        v, moves = v2, (a1, a2)
                    if v <= alpha:
                        return v, moves  # Alpha cutoff
                    beta = min(beta, v)
        return v, moves

    return max_value(state, -math.inf, math.inf, depth_limit)

def heuristic_evaluation_basic(game, state, player):
    """Evaluate the utility of a non-terminal state for the given player."""
    # Example heuristic: Count the number of empty squares as a simple heuristic
    return len(game.actions(state)) if state.to_move == player else -len(game.actions(state))
   


In [43]:
# herustic evaluation using monte carlo simulation
def heuristic_evaluation_monte_carlo(game, state, player, simulations=5):
    """Estimate the utility of a non-terminal state using Monte Carlo simulation."""
    total_score = 0
    for _ in range(simulations):
        sim_state = state
        while not game.is_terminal(sim_state):
            legal_moves = list(game.actions(sim_state))
            if len(legal_moves) < 2:
                break  # Not enough for a two-move turn
            move1 = random.choice(legal_moves)
            legal_moves.remove(move1) 
            move2 = random.choice(legal_moves)
            if move1 is not None and move2 is not None:
                sim_state = game.result(sim_state, move1, move2)
            else:
                break
        if game.is_terminal(sim_state):
            total_score += game.utility(sim_state, player)
    return total_score / simulations

In [44]:
# Implement the GreedyTwoMoveAgent and wrapper function
class GreedyTwoMoveAgent:
    def __init__(self, game, player_mark='X', gamma=0.9):
        self.game = game
        self.player_mark = player_mark
        self.gamma = gamma

    def reward(self, state):
        if self.game.is_terminal(state):
            return self.game.utility(state, self.player_mark)
        return -0.01  # Small penalty to encourage faster wins

    def act(self, game, state):
        best_score = float('-inf')
        best_move = None
        actions = list(game.actions(state))
        move_pairs = [(a, b) for i, a in enumerate(actions) for j, b in enumerate(actions) if i != j]

        for a1, a2 in move_pairs:
            try:
                next_state = game.result(state, a1, a2)
                score = self.reward(next_state)
                if score > best_score:
                    best_score = score
                    best_move = (a1, a2)
            except:
                continue
        print(f"Best move: {best_move} with score: {best_score}")   
        if best_move is None:
            raise ValueError("No valid moves available.")    
        return best_move or random.choice(move_pairs)


In [45]:
greedy_time = 0
def greedy_two_move_player(twoMoveTTT, player_mark='X'):
    """Greedy search for the best two consecutive moves."""
    global greedy_time  # Use the global variable to track time
    start_time = time.time()  # Start the timer
    move  = greedy_two_move_player_(twoMoveTTT, player_mark='X')
    elapsed_time = time.time() - start_time  # Calculate elapsed time
    greedy_time += elapsed_time  # Accumulate the time taken
    print(f"Greedy search took {elapsed_time:.4f} seconds")
    return move

# Wrapper for compatibility with play_game
def greedy_two_move_player_(twoMoveTTT, player_mark='X'):
    agent = GreedyTwoMoveAgent(twoMoveTTT, player_mark)
    def player_fn(game, state):
        return agent.act(game, state)
    return player_fn

In [46]:
import pandas as pd

def play_AI_vs_random(num_games=10, agent=minimax_search):
    game = TwoMoveTicTacToe(height=3, width=3, k=3)
    strategies = {'X': random_player, 'O':player(agent)} 
    results = {'X': 0, 'O': 0, 'D': 0}
    for _ in range(num_games):
        state = play_game(game, strategies, verbose=False)
        if state.utility == 1:
            results['X'] += 1
        elif state.utility == -1:
            results['O'] += 1
        elif state.utility == 0:
            results['D'] += 1
    return results

def play_random_vs_AI(num_games=10, agent=minimax_search):
    game = TwoMoveTicTacToe(height=3, width=3, k=3)
    strategies = {'X': player(agent), 'O': random_player} 
    results = {'X': 0, 'O': 0, 'D': 0}
    for _ in range(num_games):
        state = play_game(game, strategies, verbose=False)
        if state.utility == 1:
            results['X'] += 1
        elif state.utility == -1:
            results['O'] += 1
        elif state.utility == 0:
            results['D'] += 1
    return results


df_times = pd.DataFrame(columns=['Search_Algorithem', 'Time_Consumed', 'Random_player_win', 'AI_player_win', 'Draw'])
minimax_time = 0
results_O = play_AI_vs_random(50, minimax_search)
print("Results after 10 games:", results_O)
results_x = play_random_vs_AI(50, minimax_search)
print("Results after 10 games:", results_x)
df_times.loc[len(df_times.index)] = ['minimax', minimax_time, results_O['X']+results_x['O'], results_O['O']+results_x['X'], results_O['D']+results_x['D']]


Results after 10 games: {'X': 21, 'O': 0, 'D': 29}
Results after 10 games: {'X': 50, 'O': 0, 'D': 0}


In [47]:
#minmax with alpha beta pruning
minimax_time_alpha_beta = 0
results_O = play_AI_vs_random(50, minimax_search_alpha_beta_prune)
results_x = play_random_vs_AI(50, minimax_search_alpha_beta_prune)
df_times.loc[len(df_times.index)] = ['minimax_alpha_beta', minimax_time_alpha_beta, results_O['X']+results_x['O'], results_O['O']+results_x['X'] ,results_O['D']+results_x['D']]
print(df_times.head(10))

    Search_Algorithem  Time_Consumed  Random_player_win  AI_player_win  Draw
0             minimax     159.745266                 21             50    29
1  minimax_alpha_beta       1.997753                 16             52    32


In [48]:
#minmax with alpha beta pruning and heuristic evaluation
minimax_time_alpha_beta_prune_hct = 0

results_O = {'X': 0, 'O': 0, 'D': 0}
for _ in range(50):
    state = play_game(
                TwoMoveTicTacToe(height=3, width=3, k=3),
                dict(
                    X=random_player,
                    O=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, 3, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=10))[1],
                ),
                verbose=False
            )
    if state.utility is not None:
        if state.utility == 1:
            results_O['X'] += 1
        elif state.utility == -1:
            results_O['O'] += 1
        elif state.utility == 0:
            results_O['D'] += 1

results_X = {'X': 0, 'O': 0, 'D': 0}
for _ in range(50):
    state = play_game(
                TwoMoveTicTacToe(height=3, width=3, k=3),
                dict(
                    
                    X=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, 3, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=10))[1],
                    O=random_player,
                ),
                verbose=False
            )
    if state.utility is not None:
        if state.utility == 1:
            results_X['X'] += 1
        elif state.utility == -1:
            results_X['O'] += 1
        elif state.utility == 0:
            results_X['D'] += 1

df_times.loc[len(df_times.index)] = ['minimax_alpha_beta_prune_hct', minimax_time_alpha_beta_prune_hct, results_O['X']+results_X['O'], results_O['O']+results_X['X'], results_O['D']+results_X['D']]
print(df_times.head(10))

              Search_Algorithem  Time_Consumed  Random_player_win  \
0                       minimax     159.745266                 21   
1            minimax_alpha_beta       1.997753                 16   
2  minimax_alpha_beta_prune_hct       7.656773                 17   

   AI_player_win  Draw  
0             50    29  
1             52    32  
2             50    33  


In [49]:
random_player_time = 0
results_X = {'X': 0, 'O': 0, 'D': 0}
for _ in range(50):
    result_greedy = play_game(TwoMoveTicTacToe(), 
                            dict(X=greedy_two_move_player(TwoMoveTicTacToe(), 'X'), O=random_player), 
                            verbose=False)
    if result_greedy.utility == 1:
        results_X['X'] += 1
    elif result_greedy.utility == -1:
        results_X['O'] += 1
    elif result_greedy.utility == 0:
        results_X['D'] += 1

print("Results after 10 games:", results_X)

results_O = {'X': 0, 'O': 0, 'D': 0}
for _ in range(50):
    result_greedy = play_game(TwoMoveTicTacToe(), 
                            dict( X=random_player, O=greedy_two_move_player(TwoMoveTicTacToe(), 'O'),), 
                            verbose=False)
    if result_greedy.utility == 1:
        results_O['X'] += 1
    elif result_greedy.utility == -1:
        results_O['O'] += 1
    elif result_greedy.utility == 0:
        results_O['D'] += 1

print("Results after 10 games:", results_O)

df_times.loc[len(df_times.index)] = ['greedy', greedy_time, results_O['X']+results_X['O'], results_O['O']+results_X['X'], results_O['D']+results_X['D']]
print(df_times.head(10))

Greedy search took 0.0000 seconds
Best move: ((0, 1), (1, 2)) with score: -0.01
Best move: ((0, 0), (0, 2)) with score: 1
Greedy search took 0.0000 seconds
Best move: ((0, 1), (1, 2)) with score: -0.01
Best move: ((2, 1), (0, 0)) with score: -0.01
Greedy search took 0.0000 seconds
Best move: ((0, 1), (1, 2)) with score: -0.01
Best move: ((1, 1), (1, 0)) with score: 1
Greedy search took 0.0000 seconds
Best move: ((0, 1), (1, 2)) with score: -0.01
Best move: ((2, 1), (1, 1)) with score: 1
Greedy search took 0.0000 seconds
Best move: ((0, 1), (1, 2)) with score: -0.01
Best move: ((0, 0), (0, 2)) with score: 1
Greedy search took 0.0000 seconds
Best move: ((0, 1), (1, 2)) with score: -0.01
Best move: ((2, 1), (1, 1)) with score: 1
Greedy search took 0.0000 seconds
Best move: ((0, 1), (1, 2)) with score: -0.01
Best move: ((0, 2), (2, 2)) with score: 1
Greedy search took 0.0000 seconds
Best move: ((0, 1), (1, 2)) with score: -0.01
Best move: ((2, 1), (1, 1)) with score: 1
Greedy search took 0

In [50]:
df_times.style

,Search_Algorithem,Time_Consumed,Random_player_win,AI_player_win,Draw
0,minimax,159.745266,21,50,29
1,minimax_alpha_beta,1.997753,16,52,32
2,minimax_alpha_beta_prune_hct,7.656773,17,50,33
3,greedy,0.000000,24,50,26


In [51]:
def play_AI_AI_multi_games(num_games=1, player1=minimax_search, player2=minimax_search_alpha_beta_prune):
    game = TwoMoveTicTacToe(height=3, width=3, k=3)
    strategies = {'X': player(player1), 'O': player(player2)}
    results = {'X': 0, 'O': 0, 'D': 0}
    for _ in range(num_games):
        state = play_game(game, strategies) 
        if state.utility == 1:
            results['X'] += 1
        elif state.utility == -1:
            results['O'] += 1
        elif state.utility == 0:
            results['D'] += 1
        print(results)
    return results

def play_multi_games_str(game=TwoMoveTicTacToe(height=3, width=3, k=3), num_games=1, stratergies={'X': minimax_search, 'O': minimax_search_alpha_beta_prune}):
    game = game
    strategies = stratergies
    results = {'X': 0, 'O': 0}
    for _ in range(num_games):
        state = play_game(game, strategies)
        if state.utility == 1:
            results['X'] += 1
        elif state.utility == -1:
            results['O'] += 1
    return results

In [52]:
results_MM = play_AI_AI_multi_games(10, minimax_search, minimax_search_alpha_beta_prune)
print("Results after 10 games:", results_MM)
results_MM_alpha_beta = play_AI_AI_multi_games(10, minimax_search_alpha_beta_prune, minimax_search)
print("Results after 10 games:", results_MM_alpha_beta)

{'X': 1, 'O': 0, 'D': 0}
{'X': 2, 'O': 0, 'D': 0}
{'X': 3, 'O': 0, 'D': 0}
{'X': 4, 'O': 0, 'D': 0}
{'X': 5, 'O': 0, 'D': 0}
{'X': 6, 'O': 0, 'D': 0}
{'X': 7, 'O': 0, 'D': 0}
{'X': 8, 'O': 0, 'D': 0}
{'X': 9, 'O': 0, 'D': 0}
{'X': 10, 'O': 0, 'D': 0}
Results after 10 games: {'X': 10, 'O': 0, 'D': 0}
{'X': 1, 'O': 0, 'D': 0}
{'X': 2, 'O': 0, 'D': 0}
{'X': 3, 'O': 0, 'D': 0}
{'X': 4, 'O': 0, 'D': 0}
{'X': 5, 'O': 0, 'D': 0}
{'X': 6, 'O': 0, 'D': 0}
{'X': 7, 'O': 0, 'D': 0}
{'X': 8, 'O': 0, 'D': 0}
{'X': 9, 'O': 0, 'D': 0}
{'X': 10, 'O': 0, 'D': 0}
Results after 10 games: {'X': 10, 'O': 0, 'D': 0}


In [53]:
df_compare = pd.DataFrame(columns=['Algorithem_1', 'Algorithem_2', 'Algorithem_1_win', 'Algorithem_2_win', 'Draw'])
for d in range(1,4):
       print(f"Depth: {d}")

       results_x = {'X': 0, 'O': 0, 'D': 0}
       for _ in range(10):
              state =  play_game(TwoMoveTicTacToe(height=4, width=4, k=4), 
                                          dict( X=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, d, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=10))[1], 
                                          O=greedy_two_move_player(TwoMoveTicTacToe(height=4, width=4, k=4), 'O'),), 
                                          verbose=True)
              
              if state.utility == 1:
                     results_x['X'] += 1
              elif state.utility == -1:
                     results_x['O'] += 1
              elif state.utility == 0:
                     results_x['D'] += 1
              print("Results :", results_x)

       results_O = {'X': 0, 'O': 0, 'D': 0}
       for _ in range(10):
              state =  play_game(TwoMoveTicTacToe(height=4, width=4, k=4), 
                                          dict( X=greedy_two_move_player(TwoMoveTicTacToe(height=4, width=4, k=4), 'X'),
                                          O=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, d, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=10))[1], 
                                          ), 
                                          verbose=True)
              
              if state.utility == 1:
                     results_O['X'] += 1
              elif state.utility == -1:
                     results_O['O'] += 1
              elif state.utility == 0:
                     results_O['D'] += 1
              print("Results :", results_O)


       df_compare.loc[len(df_compare.index)] = [f"minimax_alpha_beta_prune_hct(depth={d})", 'greedy', results_x['X']+results_O['O'], results_X['O']+results_O['X'], results_x['D']+results_O['D']]
 
df_compare.style    

Depth: 1
Greedy search took 0.0000 seconds
Player X moves: (2, 2) and (1, 2)
. . . .
. . . .
. X X .
. . . .

Best move: ((0, 1), (2, 1)) with score: -0.01
Player O moves: (0, 1) and (2, 1)
. . . .
O . O .
. X X .
. . . .

Player X moves: (0, 2) and (3, 2)
. . . .
O . O .
X X X X
. . . .

Results : {'X': 1, 'O': 0, 'D': 0}
Greedy search took 0.0000 seconds
Player X moves: (3, 3) and (3, 0)
. . . X
. . . .
. . . .
. . . X

Best move: ((0, 1), (1, 2)) with score: -0.01
Player O moves: (0, 1) and (1, 2)
. . . X
O . . .
. O . .
. . . X

Player X moves: (3, 1) and (3, 2)
. . . X
O . . X
. O . X
. . . X

Results : {'X': 2, 'O': 0, 'D': 0}
Greedy search took 0.0000 seconds
Player X moves: (3, 1) and (2, 1)
. . . .
. . X X
. . . .
. . . .

Best move: ((0, 1), (1, 2)) with score: -0.01
Player O moves: (0, 1) and (1, 2)
. . . .
O . X X
. O . .
. . . .

Player X moves: (2, 0) and (2, 2)
. . X .
O . X X
. O X .
. . . .

Best move: ((0, 0), (1, 1)) with score: -0.01
Player O moves: (0, 0) and (1, 1

,Algorithem_1,Algorithem_2,Algorithem_1_win,Algorithem_2_win,Draw
0,minimax_alpha_beta_prune_hct(depth=1),greedy,15,6,0
1,minimax_alpha_beta_prune_hct(depth=2),greedy,14,1,6
2,minimax_alpha_beta_prune_hct(depth=3),greedy,20,1,0


In [54]:
df_compare = pd.DataFrame(columns=['Algorithem_1', 'Algorithem_2', 'Algorithem_1_win', 'Algorithem_2_win', 'Draw'])
for d in range(1,3):
       print(f"Depth: {d}")

       results_x = {'X': 0, 'O': 0, 'D': 0}
       for _ in range(10):
              state =  play_game(TwoMoveTicTacToe(height=4, width=4, k=4), 
                                          dict( X=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, d, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=10))[1], 
                                          O=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, d+1, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=10))[1],), 
                                          verbose=True)
              
              if state.utility == 1:
                     results_x['X'] += 1
              elif state.utility == -1:
                     results_x['O'] += 1
              elif state.utility == 0:
                     results_x['D'] += 1
              print("Results :", results_x)

       results_O = {'X': 0, 'O': 0, 'D': 0}
       
       for _ in range(10):
              state =  play_game(TwoMoveTicTacToe(height=4, width=4, k=4), 
                                          dict(
                                                X=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, d+1, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=10))[1],
                                                O=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, d, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=10))[1], 
                                               ), 
                                          verbose=True)
              
              if state.utility == 1:
                     results_O['X'] += 1
              elif state.utility == -1:
                     results_O['O'] += 1
              elif state.utility == 0:
                     results_O['D'] += 1
              print("Results :", results_O)

       y = d+1
       df_compare.loc[len(df_compare.index)] = [f"minimax_alpha_beta_prune_hct(depth={d})", f"minimax_alpha_beta_prune_hct(depth={y})", results_x['X']+results_O['O'], results_X['O']+results_O['X'], results_x['D']+results_O['D']]
 
df_compare.style   

Depth: 1
Player X moves: (3, 3) and (1, 3)
. . . .
. . . .
. . . .
. X . X

Player O moves: (0, 3) and (3, 0)
. . . O
. . . .
. . . .
O X . X

Player X moves: (1, 2) and (1, 0)
. X . O
. . . .
. X . .
O X . X

Player O moves: (1, 1) and (0, 2)
. X . O
. O . .
O X . .
O X . X

Player X moves: (0, 1) and (2, 1)
. X . O
X O X .
O X . .
O X . X

Player O moves: (0, 0) and (3, 1)
O X . O
X O X O
O X . .
O X . X

Player X moves: (2, 3) and (3, 2)
O X . O
X O X O
O X . X
O X X X

Player O moves: (2, 0) and (2, 2)
O X O O
X O X O
O X O X
O X X X

Player X moves: None and None
O X O O
X O X O
O X O X
O X X X

Results : {'X': 0, 'O': 0, 'D': 1}
Player X moves: (3, 2) and (3, 0)
. . . X
. . . .
. . . X
. . . .

Player O moves: (0, 3) and (3, 3)
. . . X
. . . .
. . . X
O . . O

Player X moves: (2, 2) and (1, 2)
. . . X
. . . .
. X X X
O . . O

Player O moves: (2, 3) and (1, 3)
. . . X
. . . .
. X X X
O O O O

Results : {'X': 0, 'O': 1, 'D': 1}
Player X moves: (0, 2) and (2, 2)
. . . .
. . . .
X . 

,Algorithem_1,Algorithem_2,Algorithem_1_win,Algorithem_2_win,Draw
0,minimax_alpha_beta_prune_hct(depth=1),minimax_alpha_beta_prune_hct(depth=2),0,4,13
1,minimax_alpha_beta_prune_hct(depth=2),minimax_alpha_beta_prune_hct(depth=3),7,5,9


In [55]:
df_compare = pd.DataFrame(columns=['Algorithem_1', 'Algorithem_2', 'Algorithem_1_win', 'Algorithem_2_win', 'Draw'])
for d in range(1,6):
       print(f"Depth: {d}")

       results_x = {'X': 0, 'O': 0, 'D': 0}
       for _ in range(10):
              state =  play_game(TwoMoveTicTacToe(height=4, width=4, k=4), 
                                          dict( X=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, 2, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=d*10))[1], 
                                          O=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, 2, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=5))[1],), 
                                          verbose=True)
              
              if state.utility == 1:
                     results_x['X'] += 1
              elif state.utility == -1:
                     results_x['O'] += 1
              elif state.utility == 0:
                     results_x['D'] += 1
              print("Results :", results_x)

       results_O = {'X': 0, 'O': 0, 'D': 0}
       for _ in range(10):
              state =  play_game(TwoMoveTicTacToe(height=4, width=4, k=4), 
                                          dict(
                                                X=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, 2, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=5))[1],
                                                O=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, 2, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=(d*10)))[1], 
                                               ), 
                                          verbose=True)
              
              if state.utility == 1:
                     results_O['X'] += 1
              elif state.utility == -1:
                     results_O['O'] += 1
              elif state.utility == 0:
                     results_O['D'] += 1
              print("Results :", results_O)

       y = 5
       x = d*10
       df_compare.loc[len(df_compare.index)] = [f"minimax_alpha_beta_prune_hct(htc_simulations={x})", f"minimax_alpha_beta_prune_hct(htc_simulations={y})", results_x['X']+results_O['O'], results_X['O']+results_O['X'], results_x['D']+results_O['D']]
 
df_compare.style   


Depth: 1
Player X moves: (2, 2) and (0, 3)
. . . .
. . . .
. . X .
X . . .

Player O moves: (0, 2) and (1, 1)
. . . .
. O . .
O . X .
X . . .

Player X moves: (3, 1) and (1, 0)
. X . .
. O . X
O . X .
X . . .

Player O moves: (3, 0) and (2, 3)
. X . O
. O . X
O . X .
X . O .

Player X moves: (0, 1) and (1, 2)
. X . O
X O . X
O X X .
X . O .

Player O moves: (2, 1) and (0, 0)
O X . O
X O O X
O X X .
X . O .

Player X moves: (1, 3) and (3, 2)
O X . O
X O O X
O X X X
X X O .

Player O moves: (3, 3) and (2, 0)
O X O O
X O O X
O X X X
X X O O

Player X moves: None and None
O X O O
X O O X
O X X X
X X O O

Results : {'X': 0, 'O': 0, 'D': 1}
Player X moves: (0, 3) and (0, 0)
X . . .
. . . .
. . . .
X . . .

Player O moves: (0, 1) and (2, 1)
X . . .
O . O .
. . . .
X . . .

Player X moves: (3, 1) and (2, 2)
X . . .
O . O X
. . X .
X . . .

Player O moves: (1, 2) and (3, 3)
X . . .
O . O X
. O X .
X . . O

Player X moves: (1, 1) and (2, 0)
X . X .
O X O X
. O X .
X . . O

Player O moves: (3, 0)

,Algorithem_1,Algorithem_2,Algorithem_1_win,Algorithem_2_win,Draw
0,minimax_alpha_beta_prune_hct(htc_simulations=10),minimax_alpha_beta_prune_hct(htc_simulations=5),0,1,20
1,minimax_alpha_beta_prune_hct(htc_simulations=20),minimax_alpha_beta_prune_hct(htc_simulations=5),1,1,19
2,minimax_alpha_beta_prune_hct(htc_simulations=30),minimax_alpha_beta_prune_hct(htc_simulations=5),0,1,20
3,minimax_alpha_beta_prune_hct(htc_simulations=40),minimax_alpha_beta_prune_hct(htc_simulations=5),0,1,20
4,minimax_alpha_beta_prune_hct(htc_simulations=50),minimax_alpha_beta_prune_hct(htc_simulations=5),0,1,20


In [57]:
df_compare = pd.DataFrame(columns=['Algorithem_1', 'Algorithem_2', 'Algorithem_1_win', 'Algorithem_2_win', 'Draw'])
for d in range(1,3):
       print(f"Depth: {d}")

       results_x = {'X': 0, 'O': 0, 'D': 0}
       for _ in range(10):
              state =  play_game(TwoMoveTicTacToe(height=4, width=4, k=4), 
                                          dict( X=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, d, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=10))[1], 
                                          O=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, d+1, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=10))[1],), 
                                          verbose=True)
              
              if state.utility == 1:
                     results_x['X'] += 1
              elif state.utility == -1:
                     results_x['O'] += 1
              elif state.utility == 0:
                     results_x['D'] += 1
              print("Results :", results_x)

       results_O = {'X': 0, 'O': 0, 'D': 0}
       
       for _ in range(10):
              state =  play_game(TwoMoveTicTacToe(height=4, width=4, k=4), 
                                          dict(
                                                X=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, d+1, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=10))[1],
                                                O=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, d, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=10))[1], 
                                               ), 
                                          verbose=True)
              
              if state.utility == 1:
                     results_O['X'] += 1
              elif state.utility == -1:
                     results_O['O'] += 1
              elif state.utility == 0:
                     results_O['D'] += 1
              print("Results :", results_O)

       y = d+1
       df_compare.loc[len(df_compare.index)] = [f"minimax_alpha_beta_prune_hct(depth={d})", f"minimax_alpha_beta_prune_hct(depth={y})", results_x['X']+results_O['O'], results_X['O']+results_O['X'], results_x['D']+results_O['D']]

print(f"Depth: 3")
results_x = {'X': 0, 'O': 0, 'D': 0}
for _ in range(10):
        state =  play_game(TwoMoveTicTacToe(height=4, width=4, k=4), 
                                    dict( X=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, 1, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=10))[1], 
                                    O=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, 3, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=10))[1],), 
                                    verbose=True)
        
        if state.utility == 1:
                results_x['X'] += 1
        elif state.utility == -1:
                results_x['O'] += 1
        elif state.utility == 0:
                results_x['D'] += 1
        print("Results :", results_x)

results_O = {'X': 0, 'O': 0, 'D': 0}
for _ in range(10):
        state =  play_game(TwoMoveTicTacToe(height=4, width=4, k=4), 
                                    dict(
                                        X=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, 3, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=10))[1],
                                        O=lambda game, state: minimax_search_alpha_beta_prune_hct(game, state, 1, lambda g, s, p: heuristic_evaluation_monte_carlo(g, s, p, simulations=10))[1], 
                                        ), 
                                    verbose=True)
        
        if state.utility == 1:
                results_O['X'] += 1
        elif state.utility == -1:
                results_O['O'] += 1
        elif state.utility == 0:
                results_O['D'] += 1
        print("Results :", results_O)


df_compare.loc[len(df_compare.index)] = [f"minimax_alpha_beta_prune_hct(depth={1})", f"minimax_alpha_beta_prune_hct(depth={3})", results_x['X']+results_O['O'], results_X['O']+results_O['X'], results_x['D']+results_O['D']]


df_compare.style   

Depth: 1
Player X moves: (2, 2) and (0, 3)
. . . .
. . . .
. . X .
X . . .

Player O moves: (1, 2) and (2, 1)
. . . .
. . O .
. O X .
X . . .

Player X moves: (3, 3) and (0, 0)
X . . .
. . O .
. O X .
X . . X

Player O moves: (0, 1) and (3, 1)
X . . .
O . O O
. O X .
X . . X

Player X moves: (1, 1) and (2, 0)
X . X .
O X O O
. O X .
X . . X

Results : {'X': 1, 'O': 0, 'D': 0}
Player X moves: (1, 1) and (1, 3)
. . . .
. X . .
. . . .
. X . .

Player O moves: (1, 2) and (2, 2)
. . . .
. X . .
. O O .
. X . .

Player X moves: (0, 2) and (0, 3)
. . . .
. X . .
X O O .
X X . .

Player O moves: (3, 3) and (0, 1)
. . . .
O X . .
X O O .
X X . O

Player X moves: (2, 1) and (3, 1)
. . . .
O X X X
X O O .
X X . O

Player O moves: (0, 0) and (2, 0)
O . O .
O X X X
X O O .
X X . O

Player X moves: (2, 3) and (1, 0)
O X O .
O X X X
X O O .
X X X O

Player O moves: (3, 2) and (3, 0)
O X O O
O X X X
X O O O
X X X O

Player X moves: None and None
O X O O
O X X X
X O O O
X X X O

Results : {'X': 1, 'O'

,Algorithem_1,Algorithem_2,Algorithem_1_win,Algorithem_2_win,Draw
0,minimax_alpha_beta_prune_hct(depth=1),minimax_alpha_beta_prune_hct(depth=2),1,6,8
1,minimax_alpha_beta_prune_hct(depth=2),minimax_alpha_beta_prune_hct(depth=3),3,6,12
2,minimax_alpha_beta_prune_hct(depth=1),minimax_alpha_beta_prune_hct(depth=3),1,11,5
